# Notebook 01: Data Collection, Cleaning, and Merging

This notebook prepares the two sources used to establish the initial monthly
Florida panel: Zillow's county-level Home Value Index and FEMA's National Risk
Index. It filters and cleans each source, validates county coverage, merges them
using county FIPS codes, and saves the merged dataset used by Notebook 02.


## 1. Setup and Raw Inputs

Define project paths, load the raw Zillow and FEMA NRI files, and confirm that
both inputs are available.


In [10]:
from pathlib import Path

import pandas as pd


def find_project_root(start_path=None):
    # Find the repository root from the current directory or its parents.
    start = Path(start_path or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "notebooks").is_dir() and (candidate / "data").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate the project root.")


PROJECT_ROOT = find_project_root()
RAW_DATA = PROJECT_ROOT / "data" / "raw"
SOURCE_PREPARATION_DIR = (
    PROJECT_ROOT
    / "data"
    / "interim"
    / "source_preparation"
)

ZILLOW_PATH = (
    RAW_DATA
    / "zillow"
    / "County_zhvi_uc_sfrcondo_tier_0.33_0.67_sm_sa_month.csv"
)
FEMA_NRI_PATH = RAW_DATA / "fema_nri" / "NRI_Table_Counties.csv"

SOURCE_PREPARATION_DIR.mkdir(parents=True, exist_ok=True)

for input_path in (ZILLOW_PATH, FEMA_NRI_PATH):
    if not input_path.exists():
        raise FileNotFoundError(f"Required input not found: {input_path}")

zillow_raw = pd.read_csv(ZILLOW_PATH)
fema_nri_raw = pd.read_csv(FEMA_NRI_PATH)

input_summary = pd.DataFrame(
    {
        "source": ["Zillow ZHVI", "FEMA NRI"],
        "rows": [len(zillow_raw), len(fema_nri_raw)],
        "columns": [zillow_raw.shape[1], fema_nri_raw.shape[1]],
    }
)

display(input_summary)


,source,rows,columns
0,Zillow ZHVI,3073,322
1,FEMA NRI,3232,465


## 2. Zillow Housing Data Preparation

The national Zillow file is restricted to Florida and monthly observations
from 2010 through 2025. The data are then reshaped from wide to long format so
that each row represents one county-month.


In [11]:
ZILLOW_IDENTIFIER_COLUMNS = [
    "RegionID",
    "SizeRank",
    "RegionName",
    "State",
    "Metro",
    "StateCodeFIPS",
    "MunicipalCodeFIPS",
]

monthly_columns = [
    column
    for column in zillow_raw.columns
    if len(column) >= 4
    and column[:4].isdigit()
    and 2010 <= int(column[:4]) <= 2025
]

zillow_florida = (
    zillow_raw.loc[
        zillow_raw["State"].eq("FL"),
        ZILLOW_IDENTIFIER_COLUMNS + monthly_columns,
    ]
    .copy()
)

ZILLOW_FILTERED_PATH = SOURCE_PREPARATION_DIR / "filtered_zhvi_data.csv"
zillow_florida.to_csv(ZILLOW_FILTERED_PATH, index=False)
zillow_florida = pd.read_csv(ZILLOW_FILTERED_PATH)

zillow_long = (
    zillow_florida.melt(
        id_vars=ZILLOW_IDENTIFIER_COLUMNS,
        value_vars=monthly_columns,
        var_name="Date",
        value_name="HousingPrice",
    )
    .assign(Date=lambda data: pd.to_datetime(data["Date"], format="%Y-%m-%d"))
    .sort_values(["RegionID", "Date"])
    .reset_index(drop=True)
)

ZILLOW_RESHAPED_PATH = SOURCE_PREPARATION_DIR / "reshaped_zhvi_data.csv"
zillow_long.to_csv(ZILLOW_RESHAPED_PATH, index=False)
zillow_long = pd.read_csv(ZILLOW_RESHAPED_PATH)
zillow_long["Date"] = pd.to_datetime(zillow_long["Date"])

zillow_structure = pd.DataFrame(
    {
        "check": [
            "Florida counties",
            "Monthly columns",
            "County-month rows",
            "Start date",
            "End date",
        ],
        "value": [
            zillow_long["RegionID"].nunique(),
            len(monthly_columns),
            len(zillow_long),
            zillow_long["Date"].min().date(),
            zillow_long["Date"].max().date(),
        ],
    }
)

display(zillow_structure)


,check,value
0,Florida counties,67
1,Monthly columns,192
2,County-month rows,12864
3,Start date,2010-01-31
4,End date,2025-12-31


### 2.1 Missing Housing Values

Missing housing values are separated into leading gaps, which occur before a
county's first valid observation, and internal gaps bounded by observed values.
This distinction determines whether a value is removed or interpolated.


In [12]:
missing_records = []

for county_name, county_data in zillow_long.groupby("RegionName", sort=True):
    missing_mask = county_data["HousingPrice"].isna()
    if not missing_mask.any():
        continue

    first_valid_date = county_data.loc[~missing_mask, "Date"].min()
    leading_missing = (
        missing_mask & county_data["Date"].lt(first_valid_date)
    ).sum()
    internal_missing = (
        missing_mask & county_data["Date"].ge(first_valid_date)
    ).sum()

    missing_records.append(
        {
            "county": county_name,
            "first_valid_date": first_valid_date.date(),
            "total_missing": int(missing_mask.sum()),
            "leading_missing": int(leading_missing),
            "internal_missing": int(internal_missing),
        }
    )

zillow_missing_summary = pd.DataFrame(missing_records)

print(f"Total missing housing values: {zillow_long['HousingPrice'].isna().sum()}")
display(zillow_missing_summary)


Total missing housing values: 98


,county,first_valid_date,total_missing,leading_missing,internal_missing
0,Dixie County,2010-01-31,1,0,1
1,Liberty County,2011-01-31,12,12,0
2,Monroe County,2016-02-29,73,73,0
3,Washington County,2011-01-31,12,12,0


### 2.2 Missing-Value Treatment and Validation

Leading gaps are removed because no earlier county observations are available
to support interpolation. The single internal gap for Dixie County is linearly
interpolated between valid surrounding months. Missing `Metro` values are
retained because they indicate counties without a reported metropolitan area.


In [13]:
valid_series_started = (
    zillow_long.groupby("RegionID")["HousingPrice"]
    .transform(lambda series: series.notna().cummax())
)

zillow_clean = zillow_long.loc[valid_series_started].copy()

internal_missing_before = int(zillow_clean["HousingPrice"].isna().sum())

zillow_clean["HousingPrice"] = (
    zillow_clean.groupby("RegionID")["HousingPrice"]
    .transform(lambda series: series.interpolate(method="linear"))
)

zillow_clean = (
    zillow_clean.sort_values(["RegionID", "Date"])
    .reset_index(drop=True)
)

duplicate_county_months = int(
    zillow_clean.duplicated(subset=["RegionID", "Date"]).sum()
)
invalid_prices = int(zillow_clean["HousingPrice"].le(0).sum())

zillow_validation = pd.DataFrame(
    {
        "check": [
            "Rows after removing leading gaps",
            "Florida counties",
            "Internal values interpolated",
            "Missing housing values",
            "Duplicate county-months",
            "Non-positive housing values",
            "Expected missing Metro values",
        ],
        "value": [
            len(zillow_clean),
            zillow_clean["RegionID"].nunique(),
            internal_missing_before,
            int(zillow_clean["HousingPrice"].isna().sum()),
            duplicate_county_months,
            invalid_prices,
            int(zillow_clean["Metro"].isna().sum()),
        ],
    }
)

if zillow_clean["RegionID"].nunique() != 67:
    raise ValueError("Expected 67 Florida counties in the cleaned Zillow data.")
if zillow_clean["HousingPrice"].isna().any():
    raise ValueError("Housing-price missing values remain after cleaning.")
if duplicate_county_months or invalid_prices:
    raise ValueError("Cleaned Zillow data failed duplicate or price validation.")

ZILLOW_CLEAN_PATH = SOURCE_PREPARATION_DIR / "cleaned_florida_zhvi_data.csv"
zillow_clean.to_csv(ZILLOW_CLEAN_PATH, index=False)

display(zillow_validation)
print(f"Saved: {ZILLOW_CLEAN_PATH.relative_to(PROJECT_ROOT)}")


,check,value
0,Rows after removing leading gaps,12767
1,Florida counties,67
2,Internal values interpolated,1
3,Missing housing values,0
4,Duplicate county-months,0
5,Non-positive housing values,0
6,Expected missing Metro values,3048


Saved: data\interim\source_preparation\cleaned_florida_zhvi_data.csv


## 3. FEMA National Risk Index Preparation

The FEMA NRI file is restricted to Florida and to the county identifiers,
population, coastal-flood risk, hurricane risk, social vulnerability, and
community resilience variables used in the study.


In [14]:
FEMA_COLUMNS = [
    "STATE",
    "STATEABBRV",
    "COUNTY",
    "STCOFIPS",
    "POPULATION",
    "CFLD_RISKS",
    "HRCN_RISKS",
    "SOVI_SCORE",
    "RESL_SCORE",
]

fema_florida = (
    fema_nri_raw.loc[
        fema_nri_raw["STATEABBRV"].eq("FL"),
        FEMA_COLUMNS,
    ]
    .copy()
)

FEMA_FILTERED_PATH = SOURCE_PREPARATION_DIR / "filtered_fema_data.csv"
fema_florida.to_csv(FEMA_FILTERED_PATH, index=False)
fema_florida = pd.read_csv(FEMA_FILTERED_PATH)

fema_missing_summary = (
    fema_florida.isna()
    .sum()
    .rename("missing_values")
    .to_frame()
)

missing_coastal_risk = fema_florida.loc[
    fema_florida["CFLD_RISKS"].isna(),
    ["COUNTY", "STCOFIPS", "CFLD_RISKS"],
]

print(f"Florida counties: {fema_florida['STCOFIPS'].nunique()}")
display(fema_missing_summary)
display(missing_coastal_risk)


Florida counties: 67


,missing_values
STATE,0
STATEABBRV,0
COUNTY,0
STCOFIPS,0
POPULATION,0
CFLD_RISKS,8
HRCN_RISKS,0
SOVI_SCORE,0
RESL_SCORE,0


,COUNTY,STCOFIPS,CFLD_RISKS
18,Gadsden,12039,NaN
22,Hamilton,12047,NaN
26,Highlands,12055,NaN
30,Jackson,12063,NaN
38,Madison,12079,NaN
48,Osceola,12097,NaN
52,Polk,12105,NaN
59,Sumter,12119,NaN


### 3.1 Coastal-Flood Risk Treatment and Validation

Eight inland counties have no coastal-flood risk score because FEMA marks the
hazard as *Not Applicable*. These values are assigned zero to represent the
absence of meaningful coastal-flood exposure rather than missing information.


In [15]:
missing_coastal_fips = fema_florida.loc[
    fema_florida["CFLD_RISKS"].isna(), "STCOFIPS"
]

coastal_risk_status = fema_nri_raw.loc[
    fema_nri_raw["STCOFIPS"].isin(missing_coastal_fips),
    ["STCOFIPS", "CFLD_RISKR"],
]

if not coastal_risk_status["CFLD_RISKR"].eq("Not Applicable").all():
    raise ValueError(
        "A missing coastal-flood risk score was not marked Not Applicable."
    )

fema_florida["CFLD_RISKS"] = fema_florida["CFLD_RISKS"].fillna(0)

duplicate_fips = int(fema_florida.duplicated(subset=["STCOFIPS"]).sum())
invalid_population = int(fema_florida["POPULATION"].le(0).sum())
remaining_missing = int(fema_florida.isna().sum().sum())

fema_validation = pd.DataFrame(
    {
        "check": [
            "Florida counties",
            "Not-applicable coastal-flood scores set to zero",
            "Remaining missing values",
            "Duplicate county FIPS codes",
            "Non-positive population values",
        ],
        "value": [
            fema_florida["STCOFIPS"].nunique(),
            len(missing_coastal_fips),
            remaining_missing,
            duplicate_fips,
            invalid_population,
        ],
    }
)

if fema_florida["STCOFIPS"].nunique() != 67:
    raise ValueError("Expected 67 Florida counties in the FEMA NRI data.")
if remaining_missing or duplicate_fips or invalid_population:
    raise ValueError("Cleaned FEMA NRI data failed validation.")

FEMA_CLEAN_PATH = SOURCE_PREPARATION_DIR / "cleaned_fema_data.csv"
fema_florida.to_csv(FEMA_CLEAN_PATH, index=False)

display(fema_validation)
print(f"Saved: {FEMA_CLEAN_PATH.relative_to(PROJECT_ROOT)}")


,check,value
0,Florida counties,67
1,Not-applicable coastal-flood scores set to zero,8
2,Remaining missing values,0
3,Duplicate county FIPS codes,0
4,Non-positive population values,0


Saved: data\interim\source_preparation\cleaned_fema_data.csv


## 4. County FIPS Alignment and Dataset Merge

State and county codes from Zillow are combined into the five-digit county FIPS
identifier. County coverage is compared before applying a many-to-one merge of
the monthly Zillow panel with the static FEMA NRI indicators.


In [16]:
zillow_clean["STCOFIPS"] = (
    zillow_clean["StateCodeFIPS"].astype(int) * 1000
    + zillow_clean["MunicipalCodeFIPS"].astype(int)
)

zillow_fips = set(zillow_clean["STCOFIPS"].unique())
fema_fips = set(fema_florida["STCOFIPS"].unique())

fips_audit = pd.DataFrame(
    {
        "check": [
            "Unique Zillow county FIPS codes",
            "Unique FEMA county FIPS codes",
            "Zillow FIPS absent from FEMA",
            "FEMA FIPS absent from Zillow",
        ],
        "value": [
            len(zillow_fips),
            len(fema_fips),
            len(zillow_fips - fema_fips),
            len(fema_fips - zillow_fips),
        ],
    }
)

if zillow_fips != fema_fips:
    raise ValueError(
        "County FIPS coverage differs between Zillow and FEMA NRI."
    )

merged_data = zillow_clean.merge(
    fema_florida,
    on="STCOFIPS",
    how="left",
    validate="many_to_one",
)

display(fips_audit)


,check,value
0,Unique Zillow county FIPS codes,67
1,Unique FEMA county FIPS codes,67
2,Zillow FIPS absent from FEMA,0
3,FEMA FIPS absent from Zillow,0


## 5. Final Validation and Export

The merged monthly panel is checked for county coverage, duplicate
county-months, missing FEMA indicators, and invalid housing values before it is
saved as the input to Notebook 02.


In [17]:
FEMA_MODEL_COLUMNS = [
    "POPULATION",
    "CFLD_RISKS",
    "HRCN_RISKS",
    "SOVI_SCORE",
    "RESL_SCORE",
]

duplicate_county_months = int(
    merged_data.duplicated(subset=["STCOFIPS", "Date"]).sum()
)
missing_fema_values = int(
    merged_data[FEMA_MODEL_COLUMNS].isna().sum().sum()
)

final_validation = pd.DataFrame(
    {
        "check": [
            "Final rows",
            "Final columns",
            "Florida counties",
            "Start date",
            "End date",
            "Duplicate county-months",
            "Missing FEMA indicator values",
            "Missing housing values",
        ],
        "value": [
            len(merged_data),
            merged_data.shape[1],
            merged_data["STCOFIPS"].nunique(),
            merged_data["Date"].min().date(),
            merged_data["Date"].max().date(),
            duplicate_county_months,
            missing_fema_values,
            int(merged_data["HousingPrice"].isna().sum()),
        ],
    }
)

if duplicate_county_months:
    raise ValueError("Duplicate county-month rows found after the merge.")
if missing_fema_values:
    raise ValueError("FEMA indicator values are missing after the merge.")
if merged_data["HousingPrice"].isna().any():
    raise ValueError("Housing values are missing after the merge.")

MERGED_OUTPUT_PATH = SOURCE_PREPARATION_DIR / "merged_florida_zillow_fema_monthly.csv"
merged_data.to_csv(MERGED_OUTPUT_PATH, index=False)

display(final_validation)
print(f"Saved: {MERGED_OUTPUT_PATH.relative_to(PROJECT_ROOT)}")


,check,value
0,Final rows,12767
1,Final columns,18
2,Florida counties,67
3,Start date,2010-01-31
4,End date,2025-12-31
5,Duplicate county-months,0
6,Missing FEMA indicator values,0
7,Missing housing values,0


Saved: data\interim\source_preparation\merged_florida_zillow_fema_monthly.csv
